# Phase 3 Exp 1: 高解像度学習(imgsz=1024)

## 仮説

ベースライン(`01_baseline_training.ipynb`)では shower クラスの mAP@0.5 が 0.42 と低く、混同行列の分析から **「他クラスとの混同ではなく、22個中20個が検出漏れ」** と判明した。

**仮説**: 入力解像度を 640 → 1024 に上げれば、shower(および他の小物体)の検出漏れが減り、特に shower の mAP@0.5 が改善する。

**理由**:
- shower 記号は間取り図内で比較的小さく描かれる
- YOLO は入力解像度に対する物体の絶対ピクセルサイズに敏感
- 解像度を上げることで小物体の特徴がより明確になる

## 実験設計(公正な比較のため、解像度以外はベースラインと完全に同一)

| 項目 | ベースライン | Exp 1 | 変更 |
|---|---|---|---|
| Model | yolov8n.pt | yolov8n.pt | 同じ |
| **imgsz** | **640** | **1024** | ⭐ 変更点 |
| epochs | 50 | 50 | 同じ |
| batch | 16 | 8 | T4 メモリ制約のため減 |
| optimizer | auto (AdamW) | auto (AdamW) | 同じ |
| seed | 42 | 42 | 同じ |
| patience | 15 | 15 | 同じ |

**注**: batch サイズだけは T4 GPU のメモリ制約で 16 → 8 に減らす(画像サイズが大きくなるため)。これは仮説検証の純度を下げるが、メモリ制約上やむを得ない。

## Section 1: 環境セットアップ

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/content/floor-plan-recognition"

if not os.path.exists(WORKDIR):
    !git clone https://github.com/Mao925/floor-plan-recognition.git {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd

In [ ]:
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
import torch
import ultralytics

print(f"PyTorch:        {torch.__version__}")
print(f"Ultralytics:    {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2: データ準備(ベースラインと同じ)

In [ ]:
from google.colab import userdata

try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print(f"✅ API キー取得成功: {ROBOFLOW_API_KEY[:3]}***{ROBOFLOW_API_KEY[-3:]}")
except Exception as e:
    print(f"❌ エラー: {e}")

In [ ]:
with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

!python scripts/download_roboflow.py

In [ ]:
!python scripts/prepare_dataset.py

## Section 3: 学習(imgsz=1024)

**所要時間予想**: T4 GPU で約 10〜20 分(解像度UP & バッチ減により、ベースライン4分より長くなる)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='data/floorplan_yolo/data.yaml',
    epochs=50,
    imgsz=1024,            # ⭐ 変更点: 640 → 1024
    batch=8,               # 解像度UPに伴いメモリ節約
    name='exp1_highres_yolov8n',
    project='runs/detect',
    patience=15,
    save=True,
    plots=True,
    device=0,
    seed=42,
)

print("\n✅ 学習完了")

In [ ]:
# 学習結果のパスを確認(Ultralytics の挙動でパスが二重になる場合あり)
!find runs -name 'best.pt' | head -5

## Section 4: 評価とベースライン比較

In [ ]:
# 学習結果のパスを取得(自動判別)
from pathlib import Path

candidates = list(Path('runs').rglob('exp1_highres_yolov8n/weights/best.pt'))
assert candidates, "best.pt が見つかりません"
best_pt = candidates[0]
results_dir = best_pt.parent.parent
print(f"学習結果フォルダ: {results_dir}")
print(f"ベストモデル:     {best_pt}")

In [ ]:
# 学習曲線などを表示
from IPython.display import Image, display

for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(f"\n=== {img_name} ===")
        display(Image(str(img_path)))

In [ ]:
# test セットで最終評価(画像サイズ=1024 で評価)
best_model = YOLO(str(best_pt))

test_metrics = best_model.val(
    data='data/floorplan_yolo/data.yaml',
    split='test',
    imgsz=1024,                       # ⭐ 学習と同じ解像度で評価
    name='exp1_test_eval',
    project='runs/detect',
)

print("\n=== Test セット全体メトリクス (Exp 1: imgsz=1024) ===")
print(f"mAP@0.5:        {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {test_metrics.box.map:.4f}")
print(f"Precision:      {test_metrics.box.mp:.4f}")
print(f"Recall:         {test_metrics.box.mr:.4f}")

In [ ]:
# クラス別 AP の表を作成 + ベースラインとの比較
import pandas as pd

class_names = ['door', 'shower', 'sink', 'staircase', 'toilet', 'window']

# ベースラインの test 結果(README に記録した値をハードコード)
baseline_ap50 = {
    'door': 0.9862, 'shower': 0.4174, 'sink': 0.8231,
    'staircase': 0.7045, 'toilet': 0.9656, 'window': 0.9924
}

# Exp 1 のクラス別 AP@0.5
exp1_ap50 = {}
for i, name in enumerate(class_names):
    ap = float(test_metrics.box.ap50[i]) if i < len(test_metrics.box.ap50) else 0
    exp1_ap50[name] = ap

# 比較表
df = pd.DataFrame({
    'Baseline (imgsz=640)': [baseline_ap50[c] for c in class_names],
    'Exp 1 (imgsz=1024)':   [exp1_ap50[c]     for c in class_names],
}, index=class_names)
df['Diff'] = df['Exp 1 (imgsz=1024)'] - df['Baseline (imgsz=640)']
df['Improvement'] = df['Diff'].apply(lambda x: '↑' if x > 0.01 else ('↓' if x < -0.01 else '−'))

print("=" * 70)
print("クラス別 mAP@0.5: ベースライン vs Exp 1")
print("=" * 70)
print(df.to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

# 全体比較
baseline_overall = 0.815
print(f"\n=== Overall mAP@0.5 ===")
print(f"  Baseline (imgsz=640):  {baseline_overall:.4f}")
print(f"  Exp 1    (imgsz=1024): {test_metrics.box.map50:.4f}")
print(f"  Diff:                  {test_metrics.box.map50 - baseline_overall:+.4f}")

### 推論サンプルの可視化(shower クラスを含むものを優先)

In [ ]:
import random
from pathlib import Path
from IPython.display import Image, display

test_images = sorted(Path('data/floorplan_yolo/test/images').glob('*.jpg'))
random.seed(42)
samples = random.sample(test_images, min(4, len(test_images)))

predict_results = best_model.predict(
    source=[str(p) for p in samples],
    imgsz=1024,
    save=True,
    project='runs/detect',
    name='exp1_samples',
    conf=0.25,
)

pred_dir = Path('runs/detect/exp1_samples')
if not pred_dir.exists():
    # nested case
    pred_dir = list(Path('runs').rglob('exp1_samples'))[0]

for img_path in sorted(pred_dir.glob('*.jpg')):
    print(f"\n=== {img_path.name} ===")
    display(Image(str(img_path)))

## Section 5: 結果の保存

In [ ]:
import shutil

src = str(results_dir)
out_zip = '/content/exp1_highres_results.zip'
shutil.make_archive(out_zip.replace('.zip', ''), 'zip', src)
print(f"✅ Zip 作成完了: {out_zip}")
!ls -lh {out_zip}

In [ ]:
from google.colab import files
files.download(out_zip)

---

## 検証結果のまとめ(実験完了後にここに記入する)

### 仮説
imgsz を 640 → 1024 に上げれば、shower の検出漏れが減り mAP@0.5 が改善する。

### 結果(↑ 上のセル出力を見て手動で記入)
- 全体 mAP@0.5: 0.815 → ?
- shower mAP@0.5: 0.417 → ?

### 解釈
- (仮説が支持された場合)解像度UPは小物体の特徴抽出を改善した
- (仮説が支持されなかった場合)別の原因(データ量不足、特徴の質)を考える必要

### 副作用
- 学習時間: 4分 → ? 分(GPU メモリ増加と計算量増加)
- 推論時間: ? ms/image(プロダクション運用上の懸念)

### 次の仮説
- 解像度UPが効くなら、さらに epochs を増やせばより良くなるか?
- 効かないなら、モデルサイズを上げる(n→s)方が良いか?